# QuantJourney SDK - VIXY Term Structure and Macro Regime

This notebook demonstrates a QuantJourney SDK workflow that combines VIX, VVIX, SKEW, VIX futures term structure, VIXY price history and macro rates to quantify carry and regime state.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
vix_raw = qj.cboe.get_vix_data(start_date='2020-01-01', end_date=END)
vvix_raw = qj.cboe.get_vvix_data(start_date='2020-01-01', end_date=END)
skew_raw = qj.cboe.get_skew_index_data(start_date='2020-01-01', end_date=END)
term_raw = qj.cboe.get_vix_term_structure()
ten_y_raw = qj.fred.get_treasury_10y(start_date='2020-01-01')
prices, volumes = price_panel(['VIXY', 'SPY', 'TLT'], start='2020-01-01', end=END)


In [ ]:
def series_from_rows(name: str, payload: Any) -> pd.Series:
    frame = pd.DataFrame(as_rows(payload))
    if frame.empty:
        return pd.Series(dtype=float, name=name)
    date_col = next((col for col in frame.columns if 'date' in str(col).lower()), frame.columns[0])
    value_col = 'close' if 'close' in frame.columns else frame.select_dtypes(include='number').columns[-1]
    frame['date'] = pd.to_datetime(frame[date_col], errors='coerce')
    frame[name] = pd.to_numeric(frame[value_col], errors='coerce')
    return frame.dropna(subset=['date', name]).set_index('date')[name].sort_index()
vol = pd.concat([series_from_rows('VIX', vix_raw), series_from_rows('VVIX', vvix_raw), series_from_rows('SKEW', skew_raw)], axis=1).dropna(how='all')
term = pd.DataFrame(as_rows(term_raw))


In [ ]:
ret = returns(prices)
carry_proxy = prices['VIXY'].pct_change(21) - prices['SPY'].pct_change(21)
regime = pd.DataFrame({'vixy_return_21d': prices['VIXY'].pct_change(21), 'spy_return_21d': prices['SPY'].pct_change(21), 'carry_proxy': carry_proxy, 'vix_level': vol['VIX'].reindex(prices.index).ffill() if 'VIX' in vol else np.nan, 'vixy_volatility_63d': ret['VIXY'].rolling(63).std() * np.sqrt(252)}).dropna(how='all')
display(pd.Series({'vix_rows': len(as_rows(vix_raw)), 'vvix_rows': len(as_rows(vvix_raw)), 'skew_rows': len(as_rows(skew_raw)), 'term_rows': len(term), 'rate_rows': len(as_rows(ten_y_raw))}))
display(regime.tail())
regime[['vixy_return_21d', 'carry_proxy', 'vixy_volatility_63d']].tail(504).plot(title='VIXY carry and volatility regime')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.